In [11]:
import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

In [12]:
MODELS_DIR = "models"
SEED = 42


def gini(y_true, y_score):
    """Gini coefficient = 2 * AUC - 1"""
    return 2.0 * roc_auc_score(y_true, y_score) - 1.0

In [13]:
test_df = pd.read_parquet("data/df_test_preprocessed.parquet")

id_cols = ["month_decision", "weekday_decision", "WEEK_NUM", "case_id"]

test_df = test_df.sort_values("WEEK_NUM", kind="mergesort").reset_index(drop=True)
week_num_test = test_df["WEEK_NUM"].copy()

test_df = test_df.drop(columns=id_cols)

X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

print(f"Test shape: {X_test.shape}")
print(f"Weeks: {week_num_test.min()} - {week_num_test.max()} "
      f"({week_num_test.nunique()} unieke weken)")
print(f"Default rate: {y_test.mean():.4f} ({int(y_test.sum())} positieven)")


Test shape: (437823, 201)
Weeks: 53 - 91 (39 unieke weken)
Default rate: 0.0328 (14376 positieven)


In [14]:
# --- Load models ---

# Logistic Regression
lr_best = joblib.load(f"{MODELS_DIR}/lr_best.joblib")

# XGBoost
xgb_best = XGBClassifier()
xgb_best.load_model(f"{MODELS_DIR}/xgb_best.json")

# MLP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp_best = torch.jit.load(f"{MODELS_DIR}/mlp_best.pt", map_location=device).eval()

mlp_weights = [v for k, v in mlp_best.state_dict().items() if k.endswith("weight")]
mlp_n_features = mlp_weights[0].shape[1]
mlp_hidden = [w.shape[0] for w in mlp_weights[:-1]]

print("LR :", f"C={lr_best.C:.6g}, l1_ratio={lr_best.l1_ratio}, "
              f"n_features={lr_best.n_features_in_}")
print("XGB:", f"n_trees={xgb_best.get_booster().num_boosted_rounds()}, "
              f"n_features={xgb_best.n_features_in_}")
print("MLP:", f"hidden={mlp_hidden}, n_features={mlp_n_features}")
print("Device:", device)


LR : C=0.000350674, l1_ratio=0.0, n_features=201
XGB: n_trees=750, n_features=201
MLP: hidden=[128], n_features=201
Device: cpu


c:\Users\Vik\Documents\UGent\Masterproef\venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
